# Optional 128-tap FIR on Arty
Results and plots below are generated only by a real board run. The reference uses the supplied 128 coefficients. AXI word transfer overhead is included in elapsed host time.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay
overlay = Overlay("../exported/lab3.bit")
fir_ip = overlay.fir_accel_0
print(fir_ip.register_map)

In [ ]:
def step(sample, clear=0):
    fir_ip.write(0x10, int(sample) & 0xffffffff)
    fir_ip.write(0x18, int(clear))
    fir_ip.write(0, 1)
    deadline = time.monotonic() + 1
    while not (fir_ip.read(0) & 2):
        if time.monotonic() > deadline:
            raise TimeoutError("FIR ap_done timeout")
    value = fir_ip.read(0x20)
    return value if value < 2**31 else value - 2**32

In [ ]:
coefficients = np.array([10, 11, 11, 8, 3, -3, -8, -11, -11, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -10, -11, -11, -8, -3, 3, 8, 11, 11, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 8, 3, -3, -8, -11, -11, -10, -10, -10, -10, -10, -10, -10, -10, -11, -11, -8, -3, 3, 8, 11, 11, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 8, 3, -3, -8, -11, -11, -10, -10, -10, -10, -10, -10, -10, -10, -11, -11, -8, -3, 3, 8, 11, 11, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10], dtype=np.int64)
assert len(coefficients) == 128
signal = np.loadtxt("fir_input.dat", dtype=np.int64)
step(0, clear=1)
start = time.perf_counter()
filtered = np.array([step(int(x)) for x in signal], dtype=np.int64)
elapsed = time.perf_counter() - start
expected = np.convolve(signal, coefficients)[:len(signal)]
np.testing.assert_array_equal(filtered, expected)
print(f"FIR PASS: {len(signal)} samples, host elapsed {elapsed:.6f} s")
np.savetxt("fir_board_output.dat", filtered, fmt="%d")

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10,6))
axes[0].plot(signal); axes[0].set_ylabel("Input")
axes[1].plot(filtered); axes[1].set_ylabel("128-tap output"); axes[1].set_xlabel("Sample")
fig.tight_layout(); fig.savefig("fir_board_plot.png", dpi=180)
plt.show()